# 01b — Driver Split Re-Audit

**SIH PS 26168 — Intelligent Dead Reckoning**

Verifies the actual driver-session structure of IO-VNBD from the live data loader.
Does NOT assume the audit is correct — reads from filesystem directly.

Outputs: `results/dataset_audit.json`

In [ ]:
import os, sys, json, datetime
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.data_loader import IOVNBDLoader

loader = IOVNBDLoader()
print(f'IOVNBDLoader initialized. Dataset root: {loader.data_root if hasattr(loader, "data_root") else "see loader"}')

# Get all session names across all splits
all_sessions = {}
for split in ['train', 'val', 'test']:
    try:
        names = loader.get_session_names(split=split)
        all_sessions[split] = names
        print(f'  {split}: {names}')
    except Exception as e:
        print(f'  {split}: ERROR — {e}')
        all_sessions[split] = []

In [ ]:
# Attempt to load each session and record metadata
audit = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'split_config': all_sessions,
    'sessions': {}
}

total_train_windows = 0
total_distance_train = 0.0

for split, names in all_sessions.items():
    for sname in names:
        print(f'Loading {sname} ({split})...', end=' ')
        try:
            sess = loader.load_session(sname, preprocess_imu=False)
            n = len(sess['accel_raw'])
            duration_s = n * 0.1  # 10 Hz
            enu = sess['enu_coords'][:, :2]
            d = float(np.sum(np.linalg.norm(np.diff(enu, axis=0), axis=1)))

            # Count potential NIO windows (w=100, stride=20)
            n_windows_train = max(0, (n - 100) // 20 + 1) if split == 'train' else 0
            total_train_windows += n_windows_train
            if split == 'train':
                total_distance_train += d

            speed = sess['gps']['speed_mps']
            mean_spd = float(np.mean(speed)) if speed is not None else -1.0

            audit['sessions'][sname] = {
                'split': split,
                'n_samples': n,
                'duration_s': round(duration_s, 1),
                'duration_min': round(duration_s / 60, 2),
                'distance_m': round(d, 1),
                'mean_speed_mps': round(mean_spd, 2),
                'n_nio_train_windows': n_windows_train,
                'status': 'OK'
            }
            print(f'OK — {n} samples, {duration_s:.0f}s, {d:.0f}m, {mean_spd:.1f}m/s')
        except Exception as e:
            audit['sessions'][sname] = {'split': split, 'status': 'ERROR', 'error': str(e)}
            print(f'ERROR — {e}')

audit['summary'] = {
    'total_sessions': sum(len(v) for v in all_sessions.values()),
    'train_sessions': len(all_sessions.get('train', [])),
    'val_sessions':   len(all_sessions.get('val',   [])),
    'test_sessions':  len(all_sessions.get('test',  [])),
    'total_nio_train_windows_w100_s20': total_train_windows,
    'total_train_distance_m': round(total_distance_train, 1),
    'driver_diversity': {
        'note': 'IO-VNBD session prefix encodes driver (M=B, Vx=E, Y=D, S=A). Verify below.',
        'train_driver_estimate': 'Drivers B (M sessions) + E (V sessions)',
        'val_driver_estimate':   'Driver D (Y sessions)',
        'test_driver_estimate':  'Driver A (S sessions) — LOCKED'
    }
}

out = PROJECT_ROOT / 'results' / 'dataset_audit.json'
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'w') as f:
    json.dump(audit, f, indent=2)

print(f'\n=== DRIVER AUDIT SUMMARY ===')
print(f'Total sessions: {audit["summary"]["total_sessions"]}')
print(f'Train: {audit["summary"]["train_sessions"]} sessions  ({total_train_windows} NIO windows  {total_distance_train/1000:.1f}km)')
print(f'Val:   {audit["summary"]["val_sessions"]} sessions')
print(f'Test:  {audit["summary"]["test_sessions"]} sessions (LOCKED - Driver A)')
print(f'Saved: {out}')
print('\n[KEY QUESTION] Does IO-VNBD contain driver identities beyond A,B,D,E?')
print('If YES: update split config and retrain with richer driver diversity.')
print('If NO:  document limitation. Do NOT fabricate drivers.')